## LSTM Loader Example

This notebook goes over how to call the load_model.py module and how to use it to load models with the correct parameters, this includes: 

1. The overall architecture (config.json)
2. The correct weights .pt or .pth
3. The correct input and target scalers (.pkl files)


#### Important Model Notes 

Model class is defined in the config.json file then called in load_model.py alongside the .pt file to create the model.

Since our input and output data are at highly different magnitudes we must always apply the input and output scalars (based on the training data) before inference. This ensures all data is treated correctly and inference is accurate. 

Important to note, scalars are normally peformed on numpy with the .cpu().numpy command while inference is performed on CUDA cores. It may be worth checking if normalization can be performed a different way.



In [17]:
# Initialize module imports used for loading and quick inference checks.
import sys
import importlib
from pathlib import Path
import torch

This is just path setup, useful if being done in the ApproxiMPC repo, but if being used in other repos or notebooks just use the path to your models directory.

In [18]:
# Setup paths and resolve which saved run to load.
# Notebook location: LSTM_training/notebooks -> project root is one level up.
project_root = Path.cwd().parent
scripts_dir = project_root / "scripts"

#Path setup to read models folder, can be altered depending on location
# Ensure Python can import the shared loader module from scripts/.
if str(scripts_dir) not in sys.path:
    sys.path.append(str(scripts_dir))

# Pick a model name and automatically select the latest dated run folder.
# Can be hard coded as needed for specific models
MODEL_NAME = "LSTM_TEST"
model_dir = project_root / "models" / MODEL_NAME
run_dirs = sorted([d for d in model_dir.iterdir() if d.is_dir()])
if not run_dirs:
    raise FileNotFoundError(f"No saved run directories found in {model_dir}")

latest_run_dir = run_dirs[-1]
config_path = latest_run_dir / f"{MODEL_NAME}_config.json"
print(f"Using config: {config_path}")

Using config: /home/devin_work/work/f1tenth/ApproxiMPC/LSTM_training/models/LSTM_TEST/20260413/LSTM_TEST_config.json


### Model Loading Starts!

In [19]:
#IMPORTANT
#How to actually import and use methods from load_model.py 

# Reload to pick up latest edits without restarting kernel.
import load_model as load_model_module
importlib.reload(load_model_module)
load_model_bundle_from_config = load_model_module.load_model_bundle_from_config

In [20]:
#Loading model and setup
#cuda setup, EASY
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

#Call bundle from config, gets every individual component for running inference in a tuple
#After this step, you are ready to run inference! Just call the modules in the correct order 
loaded_model, loaded_input_scaler, loaded_target_scaler, loaded_cfg = load_model_bundle_from_config(
    config_path=str(config_path),
    device=device,
 )

cuda


In [21]:
#Dummy Input Example
# Build a dummy sequence with the exact input size expected by the saved model.
input_dim = int(loaded_cfg["model_config"]["input_dim"])
seq_len = 100
dummy_x = torch.zeros((1, seq_len, input_dim), dtype=torch.float32, device=device)

#Model Inference 
#NOTE! both scalars are applied on CPU w/ Cuda as these are Numpy operations,
#could be worth checking if there is a better alternative

#apply input scalar to get the inputs in the correct range for the model to make accurate predicitons
input_x = loaded_input_scaler.inverse_transform(dummy_x.squeeze(0).cpu().numpy())

#loaded model to call the model and perform inference
#inference is performed on CUDA
with torch.no_grad():
    y_pred_norm = loaded_model(dummy_x)

#apply target scalar after inference to convert to real world units (rads and m/s)
y_pred_phys = loaded_target_scaler.inverse_transform(y_pred_norm.cpu().numpy())

print(f"Loaded model from: {config_path}")
print(f"Device: {device}")
print(f"Prediction (normalized): {y_pred_norm}")
print(f"Prediction (physical units): {y_pred_phys}")

Loaded model from: /home/devin_work/work/f1tenth/ApproxiMPC/LSTM_training/models/LSTM_TEST/20260413/LSTM_TEST_config.json
Device: cuda
Prediction (normalized): tensor([[ 0.1014, -0.0191]], device='cuda:0')
Prediction (physical units): [[-0.3189032  1.9614055]]
